# 04 — Running a diagnostic locally

The earlier notebooks read *pre-computed* results from the public API. This notebook does
the opposite: it runs a REF diagnostic **locally**, on your own machine, against a small
sample dataset — so you can see the evaluation pipeline end to end.

**Prerequisites:** [01 — REF concepts](01-ref-concepts.ipynb).

**What you need:** the `climate-ref` packages (installed by `uv sync`) and an internet
connection the first time, to download the sample data. The sample data is intentionally
tiny — this runs comfortably on Binder.

## 1. Fetch the sample data

The REF ships a *decimated* CMIP6 sample dataset for testing and demonstration. The
`ref_tutorials` helper wraps the REF CLI's fetch so we get it with one call.

In [ ]:
from ref_tutorials import fetch_sample_data

sample_data_dir = fetch_sample_data()
sample_data_dir

## 2. Choose a provider and diagnostic

Providers can be used directly, without the REF database or web infrastructure. We use
the bundled **example provider** and its `global-mean-timeseries` diagnostic, which
computes the annual-mean, global-mean timeseries of a dataset.

In [ ]:
import climate_ref_example
from climate_ref.config import Config

provider = climate_ref_example.provider
config = Config.default()
provider.configure(config)

diagnostic = provider.get("global-mean-timeseries")
diagnostic.slug

## 3. Build a data catalog from disk

A **data catalog** is a table of available datasets and their metadata. We can build one
directly from the sample-data directory — no database required — using the CMIP6
dataset adapter.

In [ ]:
from climate_ref.datasets import get_dataset_adapter

adapter = get_dataset_adapter("cmip6")
data_catalog = adapter.find_local_datasets(sample_data_dir / "CMIP6")

data_catalog[["source_id", "variant_label", "variable_id", "experiment_id"]].drop_duplicates()

## 4. Solve for executions

Given the diagnostic's data requirements and the available datasets, the **solver**
works out which *executions* are possible. Each execution pairs the diagnostic with one
group of datasets.

In [ ]:
from climate_ref.solver import solve_executions
from climate_ref_core.datasets import SourceDatasetType

executions = list(
    solve_executions(
        data_catalog={SourceDatasetType.CMIP6: data_catalog},
        diagnostic=diagnostic,
        provider=provider,
    )
)
print(f"{len(executions)} executions proposed")

## 5. Run one execution

An execution needs an `ExecutionDefinition` — it says which datasets to use and where to
write output. We build one for the first execution and run the diagnostic directly.

In [ ]:
from pathlib import Path

output_directory = Path("output") / "local-run"
definition = executions[0].build_execution_definition(output_directory)
definition.output_directory.mkdir(parents=True, exist_ok=True)

result = diagnostic.run(definition=definition)
assert result.successful, "diagnostic run failed"
result

## 6. Inspect the output

The run wrote an output bundle into the output directory. Let's read it back.

In [ ]:
import json

output_file = definition.to_output_path("output.json")
print(json.dumps(json.loads(output_file.read_text()), indent=2)[:2000])

## Recap

- A provider can run a diagnostic locally with no database or web infrastructure.
- `find_local_datasets` builds a data catalog straight from a directory.
- `solve_executions` turns *(diagnostic + catalog)* into concrete executions.
- `build_execution_definition` + `diagnostic.run` executes one of them.

This is the same machinery the full REF uses — just driven by hand. For testing and
debugging diagnostics this is the quickest path.

You have completed the training set. To go deeper, browse the REF documentation at
<https://climate-ref.org> and the API docs at <https://api.climate-ref.org/docs>.